## Assignment 1: Supervised Machine Learning

Welcome to the first assignment of CS 541! This assignment prepares you with some useful tools that are widely used in NLP. This assignment must be done individually.

After this assignment, you should be able to:  
1. Load a dataset from huggingface's dataset library, and do some exploratory analyses.  
2. Use scikit-learn to build and train a feature-based model.  
3. Use pytorch to build and train a feature-based model.  
4. Use Optuna to automatically search for hyperparameters.  

In CS541, any work generated by an AI shouldn't be included without declaration. If you include material generated by an AI, the level of AI use should be properly documented, and the actual tool should be noted (e.g., "I used Codex to proofread the codes and draft the analysis"). 

**AI declaration (for the reference solution):** This reference solution was drafted with the help of Claude (Anthropic) and reviewed by the TA. Students must include their own declaration in their submission; a missing declaration is a policy violation (see rubric at the end).

> **Note on the starter code.** Every cell that was provided in the starter notebook is kept **verbatim** below (same imports, same function names, same signatures, same `random_state`/seed calls). Only the `TODO` bodies are filled in, and helper functions are added *next to* the starter functions rather than by changing them. Students were not allowed to change the provided signatures either — see the rubric.

**Runtime.** `X_train` is a dense 67 349 × 3 120 matrix (≈ 1.7 GB in float64), and the provided collate function converts lists of NumPy rows to tensors, which PyTorch itself flags as slow. Expect the sklearn grid to take a few minutes per configuration and every PyTorch epoch a few minutes on CPU; the grids and the Optuna budget are kept small for that reason. All reported accuracies in this notebook come from actually running the cells (`Kernel → Restart & Run All`) — the TA copy must be run once so the numbers in the tables are filled in before it is used as the reference.

### 1. Load the dataset (5')
First, we are going to load the datasets from huggingface's `datasets` library.
Do some exploratory analysis on the dataset.  
1.1 Print out one example in the dataset. Briefly comment on what it contains.  
1.2 For each of the train, validation, and test set, compute the following statistics: 
- The number of data samples with each class label.  
- The mean and std of the sentence lengths (in words) of each `question`.  

1.3 Vectorize the validation set of the dataset, following the approaches specified in the train set example.

In [1]:
import pandas as pd 
import numpy as np 
from datasets import load_dataset  # huggingface datasets

ds = load_dataset("stanfordnlp/sst2")

c:\Users\manda\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ds

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})

#### 1.1 One example from the dataset

In [3]:
# TODO -- Print out one example in the dataset. Briefly comment on what it contains.  
example = ds["train"][0]
print(example)
print()
print("Fields:", list(example.keys()))
print("Label meaning:", ds["train"].features["label"])

{'idx': 0, 'sentence': 'hide new secretions from the parental units ', 'label': 0}

Fields: ['idx', 'sentence', 'label']
Label meaning: ClassLabel(names=['negative', 'positive'])


**Comment.** Each example has three fields:
- `sentence` — a (lower-cased, tokenised) movie-review sentence/phrase from the Stanford Sentiment Treebank, e.g. *"hide new secretions from the parental units"*;
- `label` — the binary sentiment, `0 = negative`, `1 = positive` (the `ClassLabel` feature gives the names);
- `idx` — the integer row index of the example inside its split.

So SST-2 is a binary sentence-level sentiment-classification task.

In [4]:
train_data = pd.DataFrame(ds["train"])
X_train_text = train_data["sentence"]
Y_train = train_data["label"]

val_data = pd.DataFrame(ds["validation"])
X_val_text = val_data["sentence"]
Y_val = val_data["label"]

test_data = pd.DataFrame(ds["test"])
X_test_text = test_data["sentence"]
Y_test = test_data["label"]

#### 1.2 Exploratory statistics (class counts, sentence-length mean / std)

In [5]:
# TODO -- compute the exploratory statistics
def split_stats(name, df):
    """Class counts and word-length mean/std for one split."""
    lengths = df["sentence"].str.split().str.len()          # sentence length in words
    print(f"===== {name} ({len(df)} samples) =====")
    print("Class counts:")
    print(df["label"].value_counts().sort_index().to_string())
    print(f"Sentence length (words): mean = {lengths.mean():.2f}, "
          f"std = {lengths.std(ddof=0):.2f} (population std), "
          f"{lengths.std(ddof=1):.2f} (sample std)")
    print()
    return {"split": name, "n": len(df),
            **{f"label={k}": v for k, v in df["label"].value_counts().sort_index().items()},
            "len_mean": lengths.mean(), "len_std_pop": lengths.std(ddof=0), "len_std_sample": lengths.std(ddof=1)}

stats = pd.DataFrame([split_stats("train", train_data),
                      split_stats("validation", val_data),
                      split_stats("test", test_data)]).set_index("split")
stats

===== train (67349 samples) =====
Class counts:
label
0    29780
1    37569
Sentence length (words): mean = 9.41, std = 8.07 (population std), 8.07 (sample std)

===== validation (872 samples) =====
Class counts:
label
0    428
1    444
Sentence length (words): mean = 19.55, std = 8.76 (population std), 8.76 (sample std)

===== test (1821 samples) =====
Class counts:
label
-1    1821
Sentence length (words): mean = 19.23, std = 8.92 (population std), 8.92 (sample std)



,n,label=0,label=1,len_mean,len_std_pop,len_std_sample,label=-1
split,,,,,,,
train,67349,29780.0,37569.0,9.409553,8.073746,8.073806,NaN
validation,872,428.0,444.0,19.548165,8.758873,8.763900,NaN
test,1821,NaN,NaN,19.233937,8.919936,8.922386,1821.0


**Comments.**
- The assignment text says the length of each `question`; the field in SST-2 is called `sentence`, so the statistic is computed on `sentence`.
- Train is mildly imbalanced (≈ 56 % positive / 44 % negative); validation is close to balanced.
- **The GLUE test split has hidden labels — every test label is `-1`.** A correct answer *must* notice this: the test class counts are simply `-1: 1821`, and the test set cannot be used to measure accuracy anywhere in this assignment. That is why all model selection and reporting below is done on the validation set.
- Train sentences are much shorter on average than validation/test sentences (the train split contains many sub-phrases of the parse trees, the dev/test splits contain complete sentences).
- `pandas.Series.std()` uses the sample std (`ddof=1`) whereas `numpy.std()` uses the population std (`ddof=0`); either is acceptable if the student is consistent, so both are printed.

Next we are going to vectorize the texts using TfidfVectorizer, then compute the Tf-idf features.   

In [6]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer 
from sklearn.neural_network import MLPClassifier 

counter = CountVectorizer(min_df=10, max_df=20) 
counter.fit(X_train_text)
print("Vocabulary size:", len(counter.vocabulary_))
X_train_counts = counter.transform(X_train_text)
print(X_train_counts.shape) 
count2tfidf = TfidfTransformer(use_idf=True).fit(X_train_counts)
X_train = count2tfidf.transform(X_train_counts).toarray()
print(X_train.shape)

Vocabulary size: 3120
(67349, 3120)
(67349, 3120)


#### 1.3 Vectorise the validation set with the *train-fitted* vectorizer

In [7]:
# TODO - Use the counter to convert X_val_text to occurrence vectors
# Note: don't create a new CountVectorizer, as we want to compute the vocabulary only on the train set
X_val_counts = counter.transform(X_val_text)

# TODO - use count2tfidf to transform the counts into Tfidf features
# Note: don't create a new TfidfTransformer
X_val = count2tfidf.transform(X_val_counts).toarray()

print(X_val_counts.shape, X_val.shape)
print("Validation rows with an all-zero feature vector:", int((X_val_counts.sum(axis=1) == 0).sum()),
      "of", X_val.shape[0])

(872, 3120) (872, 3120)
Validation rows with an all-zero feature vector: 369 of 872


**Comment.** Only `transform` is called — the vocabulary (`counter.vocabulary_`) and the IDF weights (`count2tfidf.idf_`) were learned on the training split and must be reused, otherwise the validation features would live in a different space and the trained model could not be applied to them. Note that with `min_df=10, max_df=20` the vocabulary only contains fairly rare words (3 120 of them), so a large fraction of validation sentences map to the all-zero vector — this caps the accuracy that any model in this assignment can reach. Students are **not** expected to change `CountVectorizer(min_df=10, max_df=20)`; it is provided code.

### 2. Train scikit-learn models (10')
Train a two-layer MLPClassifier using `random_state=0`. Manually tune the hyperparameters on the validation set. Report the procedure of hyperparameter tuning. Specifically: report the hyperparameters you have tried, and their results.  

After you are satisfied with the validation set performances, report the validation set performance. Use this set of hyperparameters and repeat the model training procedure for five times using `random_state` as 1, 2, 3, 31, 42 respectively. Record the five accuracy numbers.

**Design.** *Two-layer MLP* is read as two `Linear` layers, i.e. `hidden_layer_sizes=(h,)` — one hidden layer plus the output layer, matching the `MLP(all_layer_sizes)` class in Section 3 where `all_layer_sizes = [3120, h, 2]`. (A student who uses `hidden_layer_sizes=(h1, h2)` and *says* they interpret "two-layer" as two hidden layers is not penalised, as long as they are consistent in Sections 2 and 3.)

A helper `fit_and_eval_mlp` is defined so that the same code can be reused for tuning, for the final `random_state=0` model, and for the five-seed experiment. The starter function `train_sklearn_model` keeps its exact signature and trains the final configuration with `random_state=0`.

In [8]:
import time, warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

def fit_and_eval_mlp(X_train, Y_train, X_val, Y_val,
                     hidden_layer_sizes=(128,), learning_rate_init=1e-3, alpha=1e-4,
                     batch_size=128, max_iter=15, random_state=0, verbose=False):
    """Train one sklearn MLPClassifier and return its validation accuracy."""
    t0 = time.time()
    clf = MLPClassifier(hidden_layer_sizes=hidden_layer_sizes,
                        activation="relu",
                        solver="adam",
                        learning_rate_init=learning_rate_init,
                        alpha=alpha,
                        batch_size=batch_size,
                        max_iter=max_iter,
                        random_state=random_state,
                        verbose=verbose)
    clf.fit(X_train, Y_train)
    acc = clf.score(X_val, Y_val)
    if verbose:
        print(f"  -> val acc {acc:.4f}  ({time.time()-t0:.0f}s)")
    return acc, clf

#### 2.1 Manual hyper-parameter tuning on the validation set (`random_state=0`)

In [9]:
# Manual tuning: a small grid over hidden size, learning rate and L2 penalty (alpha), all with random_state=0.
sk_grid = [
    dict(hidden_layer_sizes=(64,),  learning_rate_init=1e-3, alpha=1e-4, batch_size=128),
    dict(hidden_layer_sizes=(128,), learning_rate_init=1e-3, alpha=1e-4, batch_size=128),
    dict(hidden_layer_sizes=(256,), learning_rate_init=1e-3, alpha=1e-4, batch_size=128),
    dict(hidden_layer_sizes=(128,), learning_rate_init=1e-2, alpha=1e-4, batch_size=128),
    dict(hidden_layer_sizes=(128,), learning_rate_init=1e-4, alpha=1e-4, batch_size=128),
    dict(hidden_layer_sizes=(128,), learning_rate_init=1e-3, alpha=1e-2, batch_size=128),
    dict(hidden_layer_sizes=(256,), learning_rate_init=1e-3, alpha=1e-2, batch_size=128),
    dict(hidden_layer_sizes=(128,), learning_rate_init=1e-3, alpha=1e-4, batch_size=32),
]

sk_results = []
for i, hp in enumerate(sk_grid, 1):
    print(f"[{i}/{len(sk_grid)}] {hp}")
    acc, _ = fit_and_eval_mlp(X_train, Y_train, X_val, Y_val, random_state=0, verbose=True, **hp)
    sk_results.append({**hp, "val_acc": acc})

sk_results_df = pd.DataFrame(sk_results)
sk_results_df

[1/8] {'hidden_layer_sizes': (64,), 'learning_rate_init': 0.001, 'alpha': 0.0001, 'batch_size': 128}
Iteration 1, loss = 0.62606467
Iteration 2, loss = 0.54016518
Iteration 3, loss = 0.52791622
Iteration 4, loss = 0.52317998
Iteration 5, loss = 0.52037763
Iteration 6, loss = 0.51858136
Iteration 7, loss = 0.51670499
Iteration 8, loss = 0.51567200
Iteration 9, loss = 0.51417731
Iteration 10, loss = 0.51320652
Iteration 11, loss = 0.51189206
Iteration 12, loss = 0.51106932
Iteration 13, loss = 0.51020256
Iteration 14, loss = 0.50927982
Iteration 15, loss = 0.50834657
  -> val acc 0.5768  (80s)
[2/8] {'hidden_layer_sizes': (128,), 'learning_rate_init': 0.001, 'alpha': 0.0001, 'batch_size': 128}
Iteration 1, loss = 0.61398040
Iteration 2, loss = 0.53816870
Iteration 3, loss = 0.52745318
Iteration 4, loss = 0.52298355
Iteration 5, loss = 0.52034065
Iteration 6, loss = 0.51815521
Iteration 7, loss = 0.51646710
Iteration 8, loss = 0.51473938
Iteration 9, loss = 0.51316405
Iteration 10, loss =

,hidden_layer_sizes,learning_rate_init,alpha,batch_size,val_acc
0,"(64,)",0.0010,0.0001,128,0.576835
1,"(128,)",0.0010,0.0001,128,0.576835
2,"(256,)",0.0010,0.0001,128,0.581422
3,"(128,)",0.0100,0.0001,128,0.581422
4,"(128,)",0.0001,0.0001,128,0.577982
5,"(128,)",0.0010,0.0100,128,0.571101
6,"(256,)",0.0010,0.0100,128,0.576835
7,"(128,)",0.0010,0.0001,32,0.584862


In [10]:
best_sk_row = sk_results_df.loc[sk_results_df["val_acc"].idxmax()]
SK_BEST = dict(hidden_layer_sizes=tuple(best_sk_row["hidden_layer_sizes"]),
               learning_rate_init=float(best_sk_row["learning_rate_init"]),
               alpha=float(best_sk_row["alpha"]),
               batch_size=int(best_sk_row["batch_size"]))
print("Chosen sklearn hyper-parameters:", SK_BEST)
print("Validation accuracy (random_state=0): {:.4f}".format(best_sk_row["val_acc"]))

Chosen sklearn hyper-parameters: {'hidden_layer_sizes': (128,), 'learning_rate_init': 0.001, 'alpha': 0.0001, 'batch_size': 32}
Validation accuracy (random_state=0): 0.5849


**Tuning report (to be written from the table above once the notebook has been run).** The table `sk_results_df` *is* the report: every configuration tried and its validation accuracy. The configuration with the highest validation accuracy is stored in `SK_BEST`. A student's report must (a) list ≥ 3 distinct configurations with their accuracies, and (b) state which one was chosen and why (highest validation accuracy).

#### 2.2 Final model (`random_state=0`) using the starter function, then five re-runs with `random_state` = 1, 2, 3, 31, 42

In [11]:
# Starter
def train_sklearn_model(X_train, Y_train, X_val, Y_val):
    # Train the two-layer MLP with the tuned hyper-parameters and random_state=0, report validation accuracy
    acc, clf = fit_and_eval_mlp(X_train, Y_train, X_val, Y_val, random_state=0, **SK_BEST)
    print("sklearn MLP (random_state=0) — validation accuracy: {:.4f}".format(acc))
    return acc

train_sklearn_model(X_train, Y_train, X_val, Y_val) 

sklearn MLP (random_state=0) — validation accuracy: 0.5849


0.5848623853211009

In [ ]:
SEEDS = [1, 2, 3, 31, 42]

sk_seed_accs = []
for s in SEEDS:
    acc, _ = fit_and_eval_mlp(X_train, Y_train, X_val, Y_val, random_state=s, **SK_BEST)
    sk_seed_accs.append(acc)
    print(f"random_state={s:>2d}: val acc = {acc:.4f}")

print("\nsklearn five-seed accuracies:", [round(a, 4) for a in sk_seed_accs])
print("mean = {:.4f}, std = {:.4f}".format(np.mean(sk_seed_accs), np.std(sk_seed_accs, ddof=1)))

random_state= 1: val acc = 0.5883
random_state= 2: val acc = 0.5837
random_state= 3: val acc = 0.5860
random_state=31: val acc = 0.5940


**Required in the write-up:** the five numbers, and their mean ± std. The spread across seeds is the "noise floor": two hyper-parameter settings whose validation accuracies differ by less than roughly one std should not be called *different* — this is exactly what Section 5 tests formally.

### 3. Train a pytorch model (10')
Here you will repeat the training of a two-layer fully-connected neural network using pytorch. Following are some specifications that may be helpful:  
- For each of the train and validation set, specify a dataloader, preferrably using `torch.utils.data.DataLoader`.  
- Use an optimizer of your choice. Adam, AdamW and SGD are popular choices.  
- Designate a number, `train_epochs`, as the number of passes through the dataset during training. Each pass through the training dataset is called an epoch.  
  - During the epoch, there may be many steps. In each step, load a batch of data from the dataloader. Compute the loss. Do a `backward()` pass to compute the gradients. Call a `step()` from the optimizer to update the model's parameters. Then zero out the gradients.
- At the end of each epoch, go through a validation run. Do *not* optimize the model during the validation run. Compute the accuracy of the model on this validation run, and print it out.

Tune the hyperparameters on the validation set. Report the hyperparameters you have tried, and their results. 

After you are satisfied with the validation set performances, record the set of hyperparameters. Use this set of hyperparameters, and repeat the model training procedure for five times using 1, 2, 3, 31, 42 as random seeds respectively. You can use `torch.manual_seed()` to set the random seeds. Record the five accuracy numbers.

**Design.** The starter classes/functions (`MLP`, `my_collate_function`, `prepare_zipped_XY`, `train_pytorch_model`) are kept with their exact signatures. Because `train_pytorch_model` hard-codes `torch.manual_seed(1)` and takes no hyper-parameter arguments, the training loop lives in a helper `run_pytorch_training(...)` that takes the seed and the hyper-parameters explicitly; `train_pytorch_model` calls it with seed 1 and the chosen configuration. The body of the helper follows the starter skeleton line by line.

Implementation notes:
- `MLP.net` is an `nn.Sequential` built from an `OrderedDict` (the starter imports `OrderedDict` for this purpose): `Linear → ReLU → … → Linear`. `all_layer_sizes = [3120, hidden, 2]` gives the two-layer network.
- `forward` returns **logits**; `nn.CrossEntropyLoss` applies the softmax internally, so no `Softmax` layer is added (adding one *and* using `CrossEntropyLoss` is a common mistake).
- Order of operations in each step: `loss.backward()` → `optim.step()` → `optim.zero_grad()` (zeroing before `backward()` is also fine; zeroing between `backward()` and `step()` is a bug).
- The validation pass uses `model.eval()` and `torch.no_grad()`; no optimiser step is taken.
- `X` is cast to `float32` before zipping to halve memory; `torch.tensor(list_of_numpy_arrays)` in the provided collate function is slow (PyTorch warns about it) — that is starter code and left untouched, which is why the grids below are kept small.

In [ ]:
# Starter
import torch
import torch.nn as nn
from collections import OrderedDict
from torch.utils.data import DataLoader

class MLP(nn.Module):
    def __init__(self, all_layer_sizes):
        super().__init__()
        layers = OrderedDict()
        n_linear = len(all_layer_sizes) - 1
        for i in range(n_linear):
            layers[f"linear{i+1}"] = nn.Linear(all_layer_sizes[i], all_layer_sizes[i+1])
            if i < n_linear - 1:                 # no activation after the output layer (logits)
                layers[f"relu{i+1}"] = nn.ReLU()
        self.net = nn.Sequential(layers)

    def forward(self, X):
        return self.net(X)                       # logits, shape (batch, n_classes)

def my_collate_function(batch):
    batch_X, batch_Y = [], []
    for item in batch:
        batch_X.append(item[0])
        batch_Y.append(item[1])
    return torch.tensor(batch_X).float(), torch.tensor(batch_Y).long()

def prepare_zipped_XY(X, Y):
    zipped = []
    for i in range(len(X)):
        zipped.append((X[i], Y[i]))
    return zipped


def run_pytorch_training(X_train, Y_train, X_val, Y_val, seed=1,
                         train_epochs=5, batch_size=128, learning_rate=1e-3, hidden_sizes=(128,),
                         verbose=True):
    """Full training procedure of the starter, parameterised by seed and hyper-parameters.
    Returns the validation accuracy after the last epoch."""
    # Define the manual seed
    torch.manual_seed(seed)

    # Set up the model, optimizer, and dataloader
    n_features = X_train.shape[1]
    n_classes = int(len(np.unique(Y_train)))
    all_layer_sizes = [n_features, *hidden_sizes, n_classes]
    model = MLP(all_layer_sizes)
    optim = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_fn = nn.CrossEntropyLoss()

    train_zipped = prepare_zipped_XY(np.asarray(X_train, dtype=np.float32), np.asarray(Y_train))
    val_zipped   = prepare_zipped_XY(np.asarray(X_val,   dtype=np.float32), np.asarray(Y_val))
    train_dataloader = DataLoader(train_zipped, batch_size=batch_size, shuffle=True,  collate_fn=my_collate_function)
    val_dataloader   = DataLoader(val_zipped,   batch_size=256,        shuffle=False, collate_fn=my_collate_function)

    if verbose:
        print("Start training!")
    last_epoch_dev_acc = 0
    for epoch in range(train_epochs):
        model.train()
        for batch_X, batch_Y in train_dataloader:
            logits = model(batch_X)                # forward pass
            loss = loss_fn(logits, batch_Y)        # compute the loss
            loss.backward()                        # compute gradients
            optim.step()                           # update parameters
            optim.zero_grad()                      # zero out the gradients

        # End-of-epoch evaluation: no optimisation, no gradient tracking
        n_correct, n_total = 0, 0
        model.eval()
        with torch.no_grad():
            for batch_X, batch_Y in val_dataloader:
                preds = model(batch_X).argmax(dim=1)
                n_correct += (preds == batch_Y).sum().item()
                n_total += batch_Y.shape[0]

        last_epoch_dev_acc = n_correct/n_total
        if verbose:
            print("Epoch {}, val accuracy {:.2f}".format(epoch+1, last_epoch_dev_acc))

    return last_epoch_dev_acc


# Hyper-parameters chosen after the manual tuning in 3.1 (updated by the tuning cell below)
PT_BEST = dict(train_epochs=5, batch_size=128, learning_rate=1e-3, hidden_sizes=(128,))

def train_pytorch_model(X_train, Y_train, X_val, Y_val):
    # Define the manual seed
    torch.manual_seed(1)

    # Define the hyperparameters
    train_epochs = PT_BEST["train_epochs"]
    batch_size = PT_BEST["batch_size"]
    learning_rate = PT_BEST["learning_rate"]
    hidden_sizes = PT_BEST["hidden_sizes"]

    # Set up the model, optimizer, dataloader and run the training loop (see run_pytorch_training)
    return run_pytorch_training(X_train, Y_train, X_val, Y_val, seed=1,
                                train_epochs=train_epochs, batch_size=batch_size,
                                learning_rate=learning_rate, hidden_sizes=hidden_sizes)

train_pytorch_model(X_train, Y_train, X_val, Y_val)

#### 3.1 Manual hyper-parameter tuning on the validation set (seed 1)

In [ ]:
pt_grid = [
    dict(train_epochs=5, batch_size=128, learning_rate=1e-3, hidden_sizes=(128,)),
    dict(train_epochs=5, batch_size=128, learning_rate=1e-2, hidden_sizes=(128,)),
    dict(train_epochs=5, batch_size=128, learning_rate=1e-4, hidden_sizes=(128,)),
    dict(train_epochs=5, batch_size=128, learning_rate=1e-3, hidden_sizes=(256,)),
    dict(train_epochs=5, batch_size=32,  learning_rate=1e-3, hidden_sizes=(128,)),
    dict(train_epochs=8, batch_size=128, learning_rate=1e-3, hidden_sizes=(64,)),
]

pt_results = []
for i, hp in enumerate(pt_grid, 1):
    print(f"[{i}/{len(pt_grid)}] {hp}")
    acc = run_pytorch_training(X_train, Y_train, X_val, Y_val, seed=1, verbose=False, **hp)
    print(f"  -> val acc {acc:.4f}")
    pt_results.append({**hp, "val_acc": acc})

pt_results_df = pd.DataFrame(pt_results)
best_pt_row = pt_results_df.loc[pt_results_df["val_acc"].idxmax()]
PT_BEST = dict(train_epochs=int(best_pt_row["train_epochs"]), batch_size=int(best_pt_row["batch_size"]),
               learning_rate=float(best_pt_row["learning_rate"]), hidden_sizes=tuple(best_pt_row["hidden_sizes"]))
print("\nChosen pytorch hyper-parameters:", PT_BEST)
print("Validation accuracy (seed 1): {:.4f}".format(best_pt_row["val_acc"]))
pt_results_df

#### 3.2 Five runs with seeds 1, 2, 3, 31, 42 using the chosen hyper-parameters

In [ ]:
pt_seed_accs = []
for s in SEEDS:
    acc = run_pytorch_training(X_train, Y_train, X_val, Y_val, seed=s, verbose=False, **PT_BEST)
    pt_seed_accs.append(acc)
    print(f"seed={s:>2d}: val acc = {acc:.4f}")

print("\npytorch five-seed accuracies:", [round(a, 4) for a in pt_seed_accs])
print("mean = {:.4f}, std = {:.4f}".format(np.mean(pt_seed_accs), np.std(pt_seed_accs, ddof=1)))

**Required in the write-up:** the table of configurations tried (`pt_results_df`), the chosen configuration, the per-epoch validation accuracies printed during training, and the five seed accuracies with mean ± std.

### 4. Hyperparameter tuning (10')
This question requires modifying your previous pytorch training scripts. Use Optuna to find the hyperparameters that can maximize the accuracy on the validation set.  

The range of hyperparameters don't need to be too large (i.e., the total program should still be runnable within a reasonable time). The most important hyperparameter is the learning rate. Other hyperparameters that you can tune include the train epochs, batch size, hidden sizes, etc.  

When you are satisfied with the hyperparameters, report the hyperparameter and the resulting validation accuracy.

**Design / the one thing students most often get wrong.** The starter creates the study with `optuna.create_study()` and that line is *provided code*. Optuna's default `direction` is **`minimize`**, so the objective must return something to be minimised — here the **validation error rate `1 − accuracy`**. Returning the accuracy directly with the default study makes Optuna search for the *worst* hyper-parameters. (Passing `direction="maximize"` and returning accuracy is an equally correct alternative; returning accuracy with the default study is a bug and is penalised.)

Search space (kept small so 20 trials finish in reasonable time):
- `learning_rate` — log-uniform in [1e-4, 1e-1] (the most important hyper-parameter, hence log scale);
- `batch_size` — categorical {64, 128, 256};
- `hidden_size` — integer in [64, 512] in steps of 64;
- `train_epochs` — integer in [2, 5].

In [ ]:
# Starter
import optuna 

def train_pytorch_model_with_optuna(trial, X_train, Y_train, X_val, Y_val):
    # Hyper-parameters are recommended by the Optuna trial instead of being fixed
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True)
    batch_size    = trial.suggest_categorical("batch_size", [64, 128, 256])
    hidden_size   = trial.suggest_int("hidden_size", 64, 512, step=64)
    train_epochs  = trial.suggest_int("train_epochs", 2, 5)

    val_acc = run_pytorch_training(X_train, Y_train, X_val, Y_val, seed=1,
                                   train_epochs=train_epochs, batch_size=batch_size,
                                   learning_rate=learning_rate, hidden_sizes=(hidden_size,),
                                   verbose=False)
    trial.set_user_attr("val_acc", val_acc)
    # optuna.create_study() minimises by default -> return the validation ERROR rate
    return 1.0 - val_acc

def find_optimal_hyper_params(X_train, Y_train, X_val, Y_val):
    # Start an Optuna study
    study = optuna.create_study()

    # objective takes only the trial; the data are captured through a closure
    def objective(trial):
        return train_pytorch_model_with_optuna(trial, X_train, Y_train, X_val, Y_val)

    study.optimize(objective, n_trials=20)

    best = study.best_trial
    print("Best trial #{}".format(best.number))
    print("Best hyper-parameters:", best.params)
    print("Best validation accuracy: {:.4f}  (objective value = error rate {:.4f})".format(
        1.0 - best.value, best.value))
    return study

optuna_study = find_optimal_hyper_params(X_train, Y_train, X_val, Y_val)

In [ ]:
# All 20 trials, sorted by validation accuracy (highest first)
trials_df = optuna_study.trials_dataframe(attrs=("number", "value", "params", "user_attrs"))
trials_df = trials_df.rename(columns={"value": "val_error"}).sort_values("val_error")
trials_df.head(20)

**Required in the write-up:** the search space, the best hyper-parameters (`study.best_params`) and the resulting validation accuracy (`1 − study.best_value`). Because Optuna's result is itself selected on the validation set, it is an optimistically biased estimate — a good answer notes that a truly held-out set would be needed to quote a final number (the SST-2 test labels are hidden, so none is available here).

### 5. Bonus: Compare the performances of the two methods (2')
Use an appropriate $t$ test, compare the five performance numbers of the sklearn model and the pytorch model *under the same set of hyperparameters*. Do their results differ?

Note: The scores for bonus will be added to the A1 total score, but the total score will be capped to 100%.

**Which test?** Two sets of five accuracies, obtained with the *same* hyper-parameters (hidden size, learning rate, batch size, number of epochs, Adam) in sklearn and in PyTorch. Although the same seed values (1, 2, 3, 31, 42) are used, a seed means something different in the two libraries (different RNGs, different initialisation and shuffling), so the two samples are **not naturally paired** — the appropriate test is an **independent two-sample t-test**. Welch's version (`equal_var=False`) is used because there is no reason to assume equal variances. A paired t-test is accepted only if the student explicitly argues that seed *k* in both libraries is a matched pair (it is a weaker argument, but not wrong).

To make the comparison fair, the sklearn model is re-trained with exactly the PyTorch hyper-parameters (`hidden_layer_sizes=(hidden,)`, `learning_rate_init=lr`, `batch_size`, `max_iter=train_epochs`, solver `adam`, `alpha=0` because the PyTorch model has no weight decay).

In [ ]:
from scipy import stats

# Re-train the sklearn model under the SAME hyper-parameters as the chosen PyTorch configuration
SK_SAME = dict(hidden_layer_sizes=tuple(PT_BEST["hidden_sizes"]),
               learning_rate_init=PT_BEST["learning_rate"],
               batch_size=PT_BEST["batch_size"],
               max_iter=PT_BEST["train_epochs"],
               alpha=0.0)

sk_same_accs = []
for s in SEEDS:
    acc, _ = fit_and_eval_mlp(X_train, Y_train, X_val, Y_val, random_state=s, **SK_SAME)
    sk_same_accs.append(acc)

print("sklearn  (same hp):", [round(a, 4) for a in sk_same_accs],
      " mean = {:.4f}, std = {:.4f}".format(np.mean(sk_same_accs), np.std(sk_same_accs, ddof=1)))
print("pytorch  (same hp):", [round(a, 4) for a in pt_seed_accs],
      " mean = {:.4f}, std = {:.4f}".format(np.mean(pt_seed_accs), np.std(pt_seed_accs, ddof=1)))

# H0: the mean validation accuracy of the two implementations is the same.
t_stat, p_val = stats.ttest_ind(sk_same_accs, pt_seed_accs, equal_var=False)   # Welch's t-test
print("\nWelch two-sample t-test: t = {:.3f}, p = {:.4f}".format(t_stat, p_val))

# For reference only (weaker pairing argument, see text above)
t_p, p_p = stats.ttest_rel(sk_same_accs, pt_seed_accs)
print("Paired t-test (reference):  t = {:.3f}, p = {:.4f}".format(t_p, p_p))

alpha_level = 0.05
if p_val < alpha_level:
    print(f"\np < {alpha_level}: reject H0 -- the two implementations give significantly different accuracies.")
else:
    print(f"\np >= {alpha_level}: fail to reject H0 -- no significant difference between the two implementations "
          "at the 5% level (with n=5 per group the test has low power, so this is not evidence of equality).")

**Interpretation (fill in the numbers after running).** State H0, the test statistic, the p-value and the decision at α = 0.05. With only five runs per group the test has very little power, so *fail to reject* should be phrased as "no evidence of a difference", not "the two are the same". Any residual difference is expected to come from implementation details (sklearn's `adam` uses `beta`/`epsilon` defaults and an internal validation split only when `early_stopping=True`; PyTorch shuffles per epoch; different initialisation schemes).

---
## Grading rubric (TA notes — delete this cell before sharing the solution with students if desired)

Total **35 points + 2 bonus** (capped at 100 %). Marking is strict: a point is awarded only if the item is *present, correct, and reported*. Code that does not run in a fresh kernel (`Kernel → Restart & Run All`) gets the output-dependent points only if the intended output can be inferred from the saved outputs; otherwise those points are 0.

### Global rules (applied before the section marks)
| Issue | Deduction |
|---|---|
| Missing AI-use declaration when AI was clearly used (or a declaration stating none while the notebook shows tell-tale generated text) | −20 % of the total, and flag to the professor |
| Provided starter code / function signatures changed (e.g. `train_sklearn_model` given extra *required* arguments, `optuna.create_study()` removed, `my_collate_function` rewritten) | −1 per changed signature (max −3). Adding *optional* keyword arguments with defaults, or adding helper functions, is fine. |
| Test set used for tuning or for reporting accuracy (impossible anyway — labels are −1 — but reporting "test accuracy = 0.xx" shows the student never looked at the data) | −2 |
| Notebook not run / no outputs saved | −2, plus loss of every "reported" point below |

### 1. Load the dataset — 5 pts
| Item | Pts | Notes |
|---|---|---|
| 1.1 Prints one example **and** comments on the three fields (`sentence`, `label` with 0/1 meaning, `idx`) | 1 | Printing without a comment: 0.5. Comment without mentioning the label semantics: 0.5. |
| 1.2 Class counts for **all three** splits | 1.5 | 0.5 per split. Test split must show `-1: 1821`; a student who "fixes" this by dropping or relabelling test rows silently loses the 0.5 for the test split. Noticing and commenting that test labels are hidden is required for full 1.5. |
| 1.2 Mean **and** std of sentence length in words for all three splits | 1.5 | 0.5 per split. Character length instead of word length: 0. Only mean or only std: half. Either `ddof` is fine. |
| 1.3 `X_val_counts = counter.transform(...)` and `X_val = count2tfidf.transform(...).toarray()` | 1 | Creating a new `CountVectorizer`/`TfidfTransformer` or calling `fit`/`fit_transform` on validation data: **0** (this is the whole point of the question). Forgetting `.toarray()` while the rest of the notebook still works: no deduction. |

### 2. scikit-learn MLP — 10 pts
| Item | Pts | Notes |
|---|---|---|
| `MLPClassifier` with a two-layer architecture and `random_state=0` for tuning | 2 | Missing `random_state=0`: −1. Not an MLP (e.g. LogisticRegression): 0 for the section. |
| Tuning procedure: ≥ 3 distinct configurations tried, each with its validation accuracy shown (table or printed lines) | 3 | 2 configs: 1.5; 1 config: 0. Only the learning rate / only one hyper-parameter varied: max 2. Configs listed without results: 1. |
| Final validation accuracy reported with the chosen hyper-parameters, and the choice justified (highest val acc) | 1 | Chosen config not the best in their own table without explanation: 0.5. |
| Five re-runs with `random_state` **= 1, 2, 3, 31, 42** exactly, all five accuracies recorded | 3 | Wrong/other seeds: max 1. Fewer than five: 0.6 each. Same hyper-parameters as the tuned ones required; changed hyper-parameters: −1. |
| Mean ± std of the five numbers (or an equivalent short comment on the variance) | 1 | |

### 3. PyTorch MLP — 10 pts
| Item | Pts | Notes |
|---|---|---|
| `MLP` class: `nn.Sequential`/layers built from `all_layer_sizes`, non-linearity between layers, `forward` returns logits | 2 | No activation between the two `Linear` layers (a linear model): −1. `Softmax` in `forward` *and* `CrossEntropyLoss`: −0.5 (works but wrong). `LogSoftmax`+`NLLLoss` is fine. |
| Train **and** validation `DataLoader`s (using `prepare_zipped_XY` + `my_collate_function` or an equivalent `Dataset`) | 1 | No shuffling of the train loader: −0.5. |
| Training step in the correct order: forward → loss → `backward()` → `step()` → `zero_grad()` | 2 | Missing `zero_grad()` (gradients accumulate): −1. `zero_grad()` between `backward()` and `step()`: −1.5. Loss not `CrossEntropyLoss`/equivalent for 2-class logits: −1. |
| End-of-epoch validation: `model.eval()` + `torch.no_grad()` (or `inference_mode`), accuracy computed and **printed every epoch** | 2 | Missing `no_grad`: −0.5. Optimiser stepped during validation: −1.5. Accuracy only printed at the end: −0.5. |
| Tuning report: ≥ 3 configurations with results | 1.5 | Same scale as Section 2. |
| Five seeds 1, 2, 3, 31, 42 via `torch.manual_seed`, five accuracies recorded with mean ± std | 1.5 | Seed set but hyper-parameters differ from the tuned ones: −0.5. Wrong seeds: max 0.5. |

### 4. Optuna — 10 pts
| Item | Pts | Notes |
|---|---|---|
| `train_pytorch_model_with_optuna` draws hyper-parameters from `trial.suggest_*` (not hard-coded) | 3 | Only one hyper-parameter suggested: 1.5. |
| Learning rate is in the search space, on a **log** scale | 1 | Linear scale over several orders of magnitude: 0.5. LR not tuned at all: 0 (the assignment says it is the most important one). |
| Optimisation **direction** is correct: returns `1 − acc` (or a loss) with the default study, or returns `acc` with `direction="maximize"` | 2 | Returns accuracy with the default (minimise) study: **0** — the search is maximising error. Also cap the "best hyper-parameters reported" item at 1 in that case, since the reported best is the worst. |
| `objective` correctly wraps the extra arguments (closure / `functools.partial` / lambda) and `study.optimize` runs 20 trials | 1 | Fewer than 20 trials without justification: 0.5. |
| Best hyper-parameters **and** the corresponding validation accuracy reported (`study.best_params`, `best_value`) | 2 | Only params or only accuracy: 1. Reporting the error rate as if it were accuracy: 1. |
| Search space is sensible and the whole study is runnable in reasonable time (bounded epochs / hidden sizes) | 1 | Unbounded / absurd ranges (e.g. epochs up to 100, hidden up to 10 000): 0. |

### 5. Bonus: t-test — 2 pts
| Item | Pts | Notes |
|---|---|---|
| Sklearn numbers regenerated under the **same** hyper-parameters as the PyTorch model (or clearly argued that they already are) | 0.5 | Comparing the Section-2 numbers (different lr/epochs/alpha) directly: 0. |
| Appropriate test: independent two-sample (Welch) t-test; paired accepted only with an explicit pairing argument | 0.5 | One-sample test, or a test on a single run: 0. |
| H0, t-statistic, p-value stated | 0.5 | |
| Correct conclusion at α = 0.05 phrased as "fail to reject / reject", with a remark on low power (n = 5) | 0.5 | "The models are the same" from p > 0.05: 0.25. |

### Typical answer patterns and how to treat them
- **Accuracy numbers.** Because the provided vectoriser (`min_df=10, max_df=20`) keeps only rare words, validation accuracies will be low-to-moderate for *every* correct implementation. Do **not** penalise low accuracy. Do penalise a student who "improves" accuracy by silently changing `CountVectorizer(min_df=10, max_df=20)` (−2 in Section 1: provided code changed and results no longer comparable with the rest of the class).
- **Runtime shortcuts.** Sub-sampling the training set to make things faster is acceptable *only* if stated explicitly and applied consistently in Sections 2–5; undocumented sub-sampling: −1 per section where it occurs.
- **Copied outputs.** If the printed five-seed accuracies are identical across seeds, the seed is not actually being applied (e.g. `torch.manual_seed` called once outside the loop, or `random_state` not passed): 0 for the five-seed item.
- **Per-epoch print missing in Section 3** but a final accuracy present: −0.5 as above; the assignment explicitly asks for it.
- When in doubt between two adjacent marks, award the lower one and write one line of justification in the feedback — that is the record you show a student who asks.